# Disease Association

This notebooks calculates cosine similarity between gene and disease embeddings. Obesity, asthama, hypertension, and schizophrenia are choosen as example cases. These diesease embeddings are compared to related and non-related gene embeddings (for each disease). Cosine similaity values are stored in cosineSimilarity/

In [ ]:
import sys
from saveAndLoad import *
import pandas as pd
import matplotlib.colors as mcolors
from matplotlib import patheffects
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from sklearn.metrics.pairwise import cosine_similarity
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
import pickle
import pandas as pd
import shap
import numpy as np

####  LOAD DATA

In [ ]:
# parse gene data
sentences = pd.read_csv('clean_genes.csv')["Summary"].tolist()
genes = pd.read_csv('clean_genes.csv')["Gene name"].tolist()
idx_to_gene = {i:gene for i,gene in enumerate(genes)}
gene_to_idx = {gene:i for i,gene in enumerate(genes)}

def parseGenes(gene_string):
    gene_string = gene_string.strip('[]')
    genes = gene_string.split(', ')
    genes = [i.strip('\'') for i in genes]
    return genes

# parse disease data
#columns ['Disease', 'Genes', 'id', 'did', 'Summary']
diseases_df = pd.read_csv('disease_mgi2_summary.csv')

disease_genes = {i:parseGenes(j) for i,j in zip(diseases_df['Disease'], diseases_df['Genes'])}
disease_summaries = {i:j for i,j in zip(diseases_df['Disease'], diseases_df['Summary'])}

diseases_of_interest = ['obesity','asthma','hypertension','schizophrenia']
assert(all([i in disease_genes.keys() for i in diseases_of_interest]))

#### LOAD MODEL

In [ ]:
#load tokenizer
model_name = "tumorailab/LitGene_ContrastiveLearning"
tokenizer = AutoTokenizer.from_pretrained(model_name)

#hyperparams
tokenizer_max_length = 512
device = 'cuda' if torch.cuda.is_available() else 'cpu'

#initialize FineTunedBERT model
model = AutoModel.from_pretrained(model_name)
model = model.to(device)

def getEmbedding(sampleInput, model, tokenizer_, tokenizer_max_length, device):
    tokens = tokenizer_.encode_plus(sampleInput, max_length = tokenizer_max_length,
                                         padding="max_length",
                                         truncation=True)
    
    formatSample = lambda x: torch.tensor(x).unsqueeze(0).to(device)

    input_ids = formatSample(tokens["input_ids"])
    mask = formatSample(tokens["attention_mask"])
    
    model.eval()
    with torch.no_grad(): 
        output = model(input_ids, mask)
        pooled_embeddings = output[0]

    pooled_embeddings = pooled_embeddings.squeeze(0)

    return pooled_embeddings.detach().cpu().numpy()

#precompute disease embeddings
pathway_embeddings = {diseases_df['Disease'][i]:getEmbedding(summary, model, tokenizer, tokenizer_max_length, device) for i,summary in enumerate(diseases_df['Summary'])}

In [ ]:
'''
Some helper functions 
'''


#compute gene embedding and compare to precomputed disease embedding
def getCosineSimilarity(gene_summary, model, tokenizer_, tokenizer_max_length, device, disease):
    gene_embedding = getEmbedding(gene_summary, model, tokenizer_, tokenizer_max_length, device)
    pathway_embedding = pathway_embeddings[disease]
    similarity = cosine_similarity(gene_embedding, pathway_embedding)[0]
    return similarity

#get contribution to cosine similarity
def getShapValues(sentences, model, tokenizer_, tokenizer_max_length,device,disease):
    predictor = lambda sampleInputs: [getCosineSimilarity(sentence,model,tokenizer_,tokenizer_max_length,device,disease) for sentence in sampleInputs]
    explainer = shap.Explainer(predictor, tokenizer_)
    shap_values = explainer(sentences) 
    return shap_values

def analyze_associated_genes(disease):
    genes = disease_genes[disease]
    sentence_indexes = [gene_to_idx[gene] for gene in genes if gene in gene_to_idx]
    print(len(genes) - len(sentence_indexes), 'genes not found of', len(genes))
    sentences_of_interest = [sentences[g] for g in sentence_indexes]
    shap_values = getShapValues(sentences_of_interest, model, tokenizer, tokenizer_max_length, device, disease)
    pickleSave(shap_values,f'shapValues/',f'shap_values_{disease}_associated.pkl')
    pickleSave(sentences_of_interest,f'shapValues/',f'sentences_{disease}_associated.pkl')
    pickleSave(sentence_indexes,f'shapValues/',f'sentence_indexes_{disease}_associated.pkl')

def analyze_predicted_genes(disease, genes_dict):
    genes = genes_dict[disease]
    sentence_indexes = [gene_to_idx[gene] for gene in genes if gene in gene_to_idx]
    print(len(genes) - len(sentence_indexes), 'genes not found of', len(genes))
    sentences_of_interest = [sentences[g] for g in sentence_indexes]
    shap_values = getShapValues(sentences_of_interest, model, tokenizer, tokenizer_max_length, device, disease)
    pickleSave(shap_values,f'shapValues/',f'shap_values_{disease}_na.pkl')
    pickleSave(sentences_of_interest,f'shapValues/',f'sentences_{disease}_na.pkl')
    pickleSave(sentence_indexes,f'shapValues/',f'sentence_indexes_{disease}_na.pkl')

def getAllCosineSimilarity(disease,in_=True):
    not_in_disease = lambda x: x not in disease_genes[disease]
    in_disease = lambda x: x in disease_genes[disease]
    f = in_disease if in_ else not_in_disease

    all_cosine_similarities = []
    for i,gene in enumerate(genes):
        if i and i%3500 == 0: print(i)
        if f(gene):
            cs = getCosineSimilarity(sentences[gene_to_idx[gene]], model, tokenizer, tokenizer_max_length, device, disease)
            all_cosine_similarities.append((gene,cs[0]))
    all_cosine_similarities = pd.DataFrame(all_cosine_similarities, columns = ['Gene','Cosine Similarity'])
    all_cosine_similarities = all_cosine_similarities.sort_values(by='Cosine Similarity',ascending=False)
    savename = 'in' if in_ else 'not_in'
    all_cosine_similarities.to_csv(f'cosineSimilarity/cosine_similarities_{savename}_{disease}.csv',index=False)
    print(f'{disease} complete')
    return all_cosine_similarities


Calculate cosine similarity between disease embeddings and embeddings of genes that are not related

In [ ]:
for disease in diseases_of_interest:
    all_cosine_similarities_not_in = getAllCosineSimilarity(disease,in_=False)

Now calculate cosine similarity for diseases and genes that are related

In [ ]:
for disease in diseases_of_interest:
    all_cosine_similarities_in = getAllCosineSimilarity(disease,in_=True)

In [ ]:
genes_not_in_disease_dict = {disease:all_cosine_similarities_not_in['Gene'].tolist()[:100] for disease,all_cosine_similarities_not_in in zip(diseases_of_interest,[pd.read_csv(f'cosineSimilarity/cosine_similarities_not_in_{disease}.csv') for disease in diseases_of_interest])}
for disease in diseases_of_interest: analyze_predicted_genes(disease,genes_not_in_disease_dict)